## Module Summary: Interactive Camera Filters
This module implements a real-time video processing loop that changes camera filters based on keyboard hotkeys. It acts as an interactive state machine, capturing live frames from the webcam, processing them mathematically through a selected filter, and rendering the output in a responsive desktop window.

### 1. Available Filter Modes
* **Preview Mode (`P`):** Shows the raw, unedited live mirror stream from the webcam.
* **Blurring Filter (`B`):** Applies a localized `13x13` pixel averaging matrix to smooth out the image and reduce noise.
* **Canny Edge Detector (`C`):** Computes structural gradients to strip away colors and isolate fine outlines and edges.
* **Corner Feature Detector (`F`):** Leverages the Shi-Tomasi tracking algorithm (`cv2.goodFeaturesToTrack`) to spot prominent corners and track them in real time with green circles.

### 2. Hotkey Control Layout
* **`P`** -> Switch to Raw Preview
* **`B`** -> Switch to Blur Filter
* **`C`** -> Switch to Canny Edge Map
* **`F`** -> Switch to Corner Feature Tracking
* **`Q` or `ESC`** -> Cleanly close the camera stream and terminate system windows.

### 3. Notebook & VS Code Optimizations
* **Bypassing `sys.argv`:** The base code used command-line parameters to find the video file. Because VS Code passes hidden network flags to run Jupyter cells, the original code thought it was opening a video file named `"-f"` and crashed instantly. Hardcoding `s = 0` forces it straight to the default webcam.
* **Global Window Cleanup:** Swapped targeted window destruction for `cv2.destroyAllWindows()` to guarantee that if a frame freezes, the entire notebook kernel doesn't hang in system memory.

In [6]:
import cv2
import numpy as np

# Define our filter state modes
PREVIEW  = 0  # Raw camera stream
BLUR     = 1  # Blurring box filter
FEATURES = 2  # Shi-Tomasi Corner Feature Detector
CANNY    = 3  # Canny Edge Detector

# Parameter dictionary for the corner detection tracking algorithm
feature_params = dict(maxCorners=500, qualityLevel=0.2, minDistance=15, blockSize=9)

# Force the video source straight to your default local webcam (0)
s = 0

image_filter = PREVIEW
alive = True
win_name = "Camera Filters"
cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)

source = cv2.VideoCapture(s)

print("Camera stream initialized!")
print("Hotkeys: P = Preview | B = Blur | C = Canny | F = Features | ESC/Q = Quit")

while alive:
    has_frame, frame = source.read()
    if not has_frame:
        print("Error: Live frame stream interrupted or unavailable.")
        break

    # Flip horizontally so the preview acts like a mirror
    frame = cv2.flip(frame, 1)

    # Process frames based on the active image_filter state
    if image_filter == PREVIEW:
        result = frame
        
    elif image_filter == CANNY:
        # Generates a binary edge-map matrix
        result = cv2.Canny(frame, 80, 150)
        
    elif image_filter == BLUR:
        # Appends a smooth localized 13x13 pixel blurring matrix
        result = cv2.blur(frame, (13, 13))
        
    elif image_filter == FEATURES:
        # Corner features must draw on top of a color frame base
        result = frame
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        corners = cv2.goodFeaturesToTrack(frame_gray, **feature_params)
        
        if corners is not None:
            # Flatten coordinates into pairs and draw small tracking markers
            for x, y in np.float32(corners).reshape(-1, 2):
                cv2.circle(result, (int(x), int(y)), 10, (0, 255, 0), 1)

    # Push processed frame matrix to system display window
    cv2.imshow(win_name, result)

    # Pull keyboard listener buffer (checks for key press every 1ms)
    key = cv2.waitKey(1)
    
    # Handle termination keys
    if key == ord("Q") or key == ord("q") or key == 27:
        alive = False
        
    # Handle filter toggle switches
    elif key == ord("C") or key == ord("c"):
        image_filter = CANNY
    elif key == ord("B") or key == ord("b"):
        image_filter = BLUR
    elif key == ord("F") or key == ord("f"):
        image_filter = FEATURES
    elif key == ord("P") or key == ord("p"):
        image_filter = PREVIEW

# Memory Lifecycle Cleanup
source.release()
cv2.destroyAllWindows()
print("Camera stream released and windows closed cleanly.")

Camera stream initialized!
Hotkeys: P = Preview | B = Blur | C = Canny | F = Features | ESC/Q = Quit
Camera stream released and windows closed cleanly.
